# DATASCI 350 - Data Science Computing

## Assignment 10 - Dependencies, environments and containers

### Instructions

This assignment evaluates your understanding of lecture 23: dependency management, `requirements.txt`, conda build strings, `uv`, images and containers, the Dockerfile, layer caching, and pushing an image to Docker Hub.

You need three things installed. Python 3, which you already have. `uv`, which you install on macOS and Linux with

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

On Windows, use the PowerShell line from the uv documentation at <https://docs.astral.sh/uv/getting-started/installation/>, or work inside WSL. And Docker Desktop, or one of the free alternatives from the "Docker Desktop and the alternatives" slide: Podman, Colima, OrbStack or Rancher Desktop. Every command below is the same under all of them. You need Docker installed before lecture 25 in any case, so this assignment brings that install forward.

Eight tasks run commands: tasks 03, 04, 05, 07, 08, 09, 10 and 12. Four are reading and reasoning about material you already have: tasks 01, 02, 06 and 11.

Tasks 07, 09, 10 and 12 all work in one folder, `a10-docker`, and build one image. Task 08 builds a second small image in its own folder. Both images start from `python:3.14-slim`, so the base image downloads once on your machine and every later build reuses it. The first build takes a couple of minutes on a slow connection and well under a minute on a fast one. The rebuilds in tasks 09 and 10 take seconds.

Task 12 pushes an image to Docker Hub, so you need a free account at <https://hub.docker.com>. Never paste a password or an access token into your submission. `docker login` prompts for them, and the transcript you paste should show only that the login succeeded.

If a `docker` command answers `Cannot connect to the Docker daemon`, Docker is installed but not running. Start the application and wait for it to settle. Appendix 02 of lecture 23 lists that error and five others you are likely to meet, with the fix for each.

You must complete this assignment individually. You may use notes, books, and AI tools, but you must submit original work. I strongly recommend that you do not use AI tools to generate your solutions, as this will not help you learn the material. Acknowledge all resources used, including input from classmates and AI, in a short note at the end of your submission. If you are unsure about permissible resources or proper acknowledgement, please ask the instructor.

### Submission

Please submit your solutions as either a single Jupyter notebook or a PDF file. For each task, paste the commands you ran, any code you wrote, the output you got, and your written answer where the task asks for one. Screenshots are welcome but not required if you paste the text. Submit your completed assignment to Canvas.

### Task 01

A classmate has pushed their project to GitHub. The repository is 300 MB, because they committed the whole `.venv/` folder along with their code. Their commit message says "add .venv so the project is reproducible".

Answer in one paragraph. Say why committing the folder does not make the project reproducible. Say what breaks for a collaborator who clones the repository onto a different operating system, and be specific about what is inside `.venv/` that will not work there. Then name the one file that belongs in git instead, and give the exact line they should add to `.gitignore`.

Hint: the "`venv` and `pip`: an isolated Python" slide of lecture 23 says what the folder holds and what goes into git.

*Commands, output, and answer here.*

### Task 02

Lecture 23 took a conda package specification apart with a small diagram. Here is the shape of it in words: a full conda specification is three fields joined by `=` signs, and the last of the three is itself made of two pieces, which is why the slide labels four parts in all.

Here is a real line from a full conda export:

```
scipy=1.16.2=py314h9ae6f1c_1
```

Answer three things.

1. Name the four parts and say which piece of the line each one is.
2. Say which single part makes this line fail to install on a collaborator's machine, and explain what about that part is tied to one machine.
3. Say what `conda env export --no-builds` does to the line, and write out what the line looks like afterwards.

Hint: the "What are build strings?" slide of lecture 23 labels the four parts of `numpy=1.21.5=py39h12345_0` and lists what the third field encodes.

*Commands, output, and answer here.*

### Task 03

Make a new folder for this task, anywhere outside your other projects, and work inside it.

```bash
python3 -m venv .venv
source .venv/bin/activate    # macOS and Linux
.venv\Scripts\activate       # Windows
pip install requests polars
```

Now paste three things:

1. The output of `which python` (`where python` on Windows) and of `pip --version`.
2. The line `pip install` printed starting `Successfully installed`.
3. The file you get from

```bash
pip freeze > requirements.txt
cat requirements.txt
```

Then answer two things.

1. You asked for two packages and the file has more lines than that. Count the lines, say why there are that many, and name one package in the file that you did not ask for and say which of your two packages pulled it in.
2. Every line uses `==`. Say what `==` guarantees for the person who runs `pip install -r requirements.txt` from your file, and name one thing it does not guarantee.

Hint: the "`requirements.txt`: the file you commit" slide of lecture 23 runs the same three commands and gets seven lines from two packages.

*Commands, output, and answer here.*

### Task 04

The same job again, with `uv` instead of `venv` and `pip`. Work outside the folder from task 03 and outside any existing git repository, because `uv init` creates a new one.

```bash
uv init a10-uv-demo --python 3.14
cd a10-uv-demo
uv add pandas requests
```

Paste two things: everything `uv add` printed, and the contents of `pyproject.toml`.

Then answer three things.

1. Report the resolve time from the `Resolved N packages in ...` line, and the number of packages it resolved.
2. `uv add` printed a line about creating a virtual environment. Say what that tells you about a step you had to do by hand in task 03 and did not have to do here.
3. Compare `pyproject.toml` with the `requirements.txt` from task 03 in a few sentences. Both record your dependencies. Say what is in one and not the other, and which of the two records what you asked for rather than what you ended up with.

Hint: the "`uv`: a fast Python package manager" slide of lecture 23 runs `uv init` and `uv add`, and the "`uv`: project files and sharing" slide shows the `pyproject.toml` that comes out.

*Commands, output, and answer here.*

### Task 05

Stay in `a10-uv-demo`. A Dockerfile cannot read `pyproject.toml` and `uv.lock`, so `uv` writes the file it can read.

```bash
uv export --format requirements-txt --no-hashes
```

Paste the whole output.

Then answer two things.

1. Most lines have a `# via` comment under them. Say what those comments tell you, pick two lines from your own output and read them out in a sentence each, and say why `pip freeze` in task 03 could never produce them.
2. Now redirect it into the file a Dockerfile copies:

```bash
uv export --format requirements-txt --no-hashes > requirements.txt
```

You now have three dependency files in the project: `pyproject.toml`, `uv.lock` and `requirements.txt`. Say in a few sentences why the Dockerfile wants this generated `requirements.txt` rather than `uv.lock`, even though `uv.lock` is the more precise of the two.

Hint: the "`uv`: handing a project to Docker" slide of lecture 23 runs this export and explains the `# via` comments.

*Commands, output, and answer here.*

### Task 06

A project group has a working Dockerfile. Its install step is the usual one:

```dockerfile
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
```

They manage their dependencies with `uv`, so they also have a `pyproject.toml` and a `uv.lock`. They are arguing about what to commit: `requirements.txt` only, `pyproject.toml` and `uv.lock` only, or all three. Nobody in the group wants to edit two files every time they add a package.

Recommend one of the three options in one paragraph. Your answer must deal with two people. First the marker, who clones the repository and runs `docker build` and nothing else. Second a teammate who adds a package next week and has to keep the repository consistent. Say what each of them needs to be in git, and if your answer creates a step somebody has to remember, name the step.

Hint: the "How to declare dependencies?" slide of lecture 23 says which file a Dockerfile reads, and the two `uv` slides say which files belong in git and which command regenerates the third.

*Commands, output, and answer here.*

### Task 07

This task builds the lecture 23 demo image. Tasks 09, 10 and 12 keep working in the same folder, so pick somewhere you can find again.

Make a folder called `a10-docker`, open a terminal inside it, and download the three files:

```bash
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-23/docker/Dockerfile
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-23/docker/hello.py
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-23/docker/requirements.txt
```

Check that `ls` shows all three, then build and run:

```bash
docker build --progress=plain -t a10-example .
docker run --rm a10-example
docker images a10-example
```

The `--progress=plain` flag prints every step and its timing instead of collapsing them, which is what the rest of this assignment reads.

Paste three things: the build output, the three lines `docker run` printed, and the size from `docker images`. The `pip install` step prints a download progress bar for each package; you may cut those lines, but keep every line that starts with `#`, because those are the step and timing lines.

Then answer in a few sentences: name the step that took longest on your machine, say what it was doing, and say whether it will take that long again the next time you build.

Hint: the "The build, step by step" slide of lecture 23 shows the same output with the five numbered steps and their timings.

*Commands, output, and answer here.*

### Task 08

Now write a Dockerfile of your own, for a different package. Make a second folder called `a10-duck`, separate from `a10-docker`, and put two files in it.

`summary.py`:

```python
import duckdb

duckdb.sql("SELECT 42 AS answer, version() AS duckdb").show()
```

`requirements.txt`, one line:

```
duckdb==1.5.5
```

Write the `Dockerfile` yourself. It has five instructions, in the pattern from the "The Dockerfile" slide: start from `python:3.14-slim`, work in `/app`, copy the requirements, install them with `pip install --no-cache-dir -r requirements.txt`, copy the script, and set the script as the command the container runs. Two of those instructions are `COPY` lines, and the order you put them in decides whether the install stays cached later, so think about which one goes first.

Build it and run it:

```bash
docker build --progress=plain -t a10-duck .
docker run --rm a10-duck
```

Paste three things: your `Dockerfile`, the last few lines of the build including the `pip install` step, and what `docker run` printed. The run output is a small table with one row.

Then answer in a few sentences: say why your `COPY requirements.txt .` line sits above `COPY summary.py .`, and say what would go wrong on every later build if you swapped them.

Hint: the "The Dockerfile" slide of lecture 23 has the five instructions, and the "Layer caching" slide explains the ordering.

*Commands, output, and answer here.*

### Task 09

Back in `a10-docker`. Open `requirements.txt` and add one line:

```
requests==2.32.5
```

Change nothing else. Then build again, with the same tag:

```bash
docker build --progress=plain -t a10-example .
```

Paste the build output, keeping every line that starts with `#`.

Then answer three things.

1. List the five numbered steps and say for each one whether it says `CACHED` or ran again.
2. Only one file changed. Explain why more than one step had to rerun.
3. State the caching rule in one sentence, in a form that would let you predict the answer to question 1 for any Dockerfile.

The container prints the same three lines as before, because you did not touch `hello.py`. That is expected, and this task does not ask you to run it.

Hint: the "Layer caching" slide of lecture 23 says when Docker reuses a layer and what happens to the layers below one that misses.

*Commands, output, and answer here.*

### Task 10

Still in `a10-docker`, and leaving `requirements.txt` exactly as you left it in task 09. Open `hello.py` and change the greeting so it includes your own name, for example

```python
print("Hello, DATASCI350, from Ada!")
```

Change nothing else. Build again and run it:

```bash
docker build --progress=plain -t a10-example .
docker run --rm a10-example
```

Paste the build output and the run output. Report the total build time.

Then answer in a few sentences. Say which steps said `CACHED` this time and which reran, and contrast that with task 09: you edited one file in each task, so say what made the difference. Then say what this means for your project. You will edit the report inside the container dozens of times in the last week of the semester, and each edit is a rebuild.

Hint: the "Layer caching" slide of lecture 23 explains why `COPY requirements.txt` sits above the script, in terms of exactly this edit cycle.

*Commands, output, and answer here.*

### Task 11

The final project rubric in `project-instructions.qmd` gives one line 30% of the grade:

> Reproducibility: the container builds and the report renders from a clean `docker run`, 30%

The project instructions also say what that means in practice: "If `docker run` on my machine does not reproduce your report, your project does not exist."

The week before the deadline, your group borrows a laptop that has never seen your project. It has Docker installed and nothing else: no Python, no conda environment, none of your data, none of your packages.

Write a five-item checklist to run on that laptop, in the order you would run it. Each item is one command or one action, with one line saying what a failure of that item would tell you. Your five items must get from an empty machine to a rendered report, and at least one of them must catch a file that exists on your own machine but was never committed.

Hint: the "Docker pull" slide of lecture 23 says what the marker actually runs, and the project instructions list the four steps of the workflow the container has to reproduce.

*Commands, output, and answer here.*

### Task 12

This task shares the image from task 10. You need a free Docker Hub account: sign up at <https://hub.docker.com> if you do not have one.

Log in from the terminal, tag the image with your own username, and push it. Replace `<username>` with your Docker Hub username in both lines:

```bash
docker login
docker tag a10-example <username>/a10-example:latest
docker push <username>/a10-example:latest
```

Paste two things: what `docker push` printed, and the URL of your image on Docker Hub, which is `https://hub.docker.com/r/<username>/a10-example`. Paste the confirmation line from `docker login`, and nothing else from it. Do not paste your password or your access token anywhere in your submission.

Then answer two things.

1. Somebody with no Python on their machine, no NumPy and no pandas, wants to run your image. Say what they do need installed, and write the two commands they would type.
2. Your `hello.py` prints the NumPy and pandas versions. Say why those two numbers are the same on their machine as on yours, and name where those versions were fixed.

Hint: the "Let's run and share the container" and "Docker pull" slides of lecture 23 have all four commands.

*Commands, output, and answer here.*